<a href="https://colab.research.google.com/github/Geberty/USTspecialTopics2026spring/blob/main/assignment4cnn_dna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Wang R, Wang Z, Wang J, et al. SpliceFinder: ab initio prediction of splice sites using convolutional neural network[J]. BMC bioinformatics, 2019, 20(23): 1-13.

In [1]:
import numpy as np
import requests
from io import StringIO

# label.txt
labelurl = 'https://drive.google.com/uc?export=download&id=1OMwMKG-Hpkst_GQG7wbX_JxyHmRwF0y7'
# Note: For direct links, use the format: https://drive.google.com/uc?export=download&id=FILE_ID
response = requests.get(labelurl)
labels = np.loadtxt(StringIO(response.text))

# encodesequence.txt
encodeurl = 'https://drive.google.com/uc?export=download&id=136Yc5ye3Mlqd73vfe6RMEc8pldI18FzO'
# Note: For direct links, use the format: https://drive.google.com/uc?export=download&id=FILE_ID
response = requests.get(encodeurl)
encoded_seq = np.loadtxt(StringIO(response.text))

In [3]:
# ON GPU
# Test different window length
from __future__ import print_function
import numpy as np
import time
import keras
from keras.models import Sequential
from keras.layers import Flatten
from keras.layers import Dense, Dropout, Activation
from keras.optimizers import Adam
from keras.optimizers import RMSprop
from keras.callbacks import ModelCheckpoint
from keras.applications import *
from sklearn.model_selection import train_test_split
from keras.callbacks import EarlyStopping
from keras.utils import to_categorical

Length = 400  # length of window

def load_data():

    #labels = np.loadtxt('label.txt')
    #encoded_seq = np.loadtxt('encoded_seq.txt')
    encoded_seq_choose = encoded_seq[:, ((400-Length)*2):(1600-(400-Length)*2)]

    print(encoded_seq_choose.shape)
    x_train,x_test,y_train,y_test = train_test_split(encoded_seq_choose,labels,test_size=0.2)

    return np.array(x_train),np.array(y_train),np.array(x_test),np.array(y_test)


def cnn_classifier():

    model = Sequential()


    model.add(keras.layers.Conv1D(filters=50, kernel_size=9, strides=1, padding='same', input_shape=(Length, 4), activation='relu'))

    model.add(Flatten())
    model.add(Dense(100,activation='relu'))

    model.add(Dropout(0.3))
    model.add(Dense(3,activation='softmax'))

    adam = Adam(learning_rate=1e-4)
    model.compile(optimizer=adam, loss='categorical_crossentropy', metrics=['accuracy'])

    return model


def training_process(x_train,y_train,x_test,y_test):

    x_train = x_train.reshape(-1, Length, 4)
    y_train = to_categorical(y_train, num_classes=3)
    x_test = x_test.reshape(-1, Length, 4)
    y_test = to_categorical(y_test, num_classes=3)

    early_stopping = EarlyStopping(monitor='val_loss', patience=3, verbose=1, restore_best_weights=True)

    print("======================")
    print('Convolution Neural Network')
    start_time = time.time()
    model = cnn_classifier()
    model.fit(x_train, y_train, epochs=40, batch_size=50, validation_data=(x_test, y_test), callbacks=[early_stopping])
    #model.save('CNN.h5')
    loss,accuracy = model.evaluate(x_test,y_test)
    print('testing accuracy: {}'.format(accuracy))
    print('training on GPU took %fs'%(time.time()-start_time))


def main():
    x_train,y_train,x_test,y_test = load_data()

    training_process(x_train,y_train,x_test,y_test)


if __name__ == '__main__':
    main()

(6695, 1600)
Convolution Neural Network
Epoch 1/40


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


108/108 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.9615 - loss: 0.1064 - val_accuracy: 0.9985 - val_loss: 0.0142
Epoch 2/40
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9985 - loss: 0.0131 - val_accuracy: 0.9985 - val_loss: 0.0123
Epoch 3/40
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9974 - loss: 0.0163 - val_accuracy: 0.9985 - val_loss: 0.0129
Epoch 4/40
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9974 - loss: 0.0136 - val_accuracy: 0.9985 - val_loss: 0.0130
Epoch 5/40
108/108 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9978 - loss: 0.0077 - val_accuracy: 0.9985 - val_loss: 0.0126
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9984 - loss: 0.0136
testing accuracy: 0.9985063672065735
training on GPU took 7.619514s


In [ ]:
# changed runtime to CPU only

In [2]:
# ON CPU
# Test different window length
from __future__ import print_function
import numpy as np
import time
import keras
from keras.models import Sequential
from keras.layers import Flatten
from keras.layers import Dense, Dropout, Activation
from keras.optimizers import Adam
from keras.optimizers import RMSprop
from keras.callbacks import ModelCheckpoint
from keras.applications import *
from sklearn.model_selection import train_test_split
from keras.callbacks import EarlyStopping
from keras.utils import to_categorical

Length = 400  # length of window

def load_data():

    #labels = np.loadtxt('label.txt')
    #encoded_seq = np.loadtxt('encoded_seq.txt')
    encoded_seq_choose = encoded_seq[:, ((400-Length)*2):(1600-(400-Length)*2)]

    print(encoded_seq_choose.shape)
    x_train,x_test,y_train,y_test = train_test_split(encoded_seq_choose,labels,test_size=0.2)

    return np.array(x_train),np.array(y_train),np.array(x_test),np.array(y_test)


def cnn_classifier():

    model = Sequential()


    model.add(keras.layers.Conv1D(filters=50, kernel_size=9, strides=1, padding='same', input_shape=(Length, 4), activation='relu'))

    model.add(Flatten())
    model.add(Dense(100,activation='relu'))

    model.add(Dropout(0.3))
    model.add(Dense(3,activation='softmax'))

    adam = Adam(learning_rate=1e-4)
    model.compile(optimizer=adam, loss='categorical_crossentropy', metrics=['accuracy'])

    return model


def training_process(x_train,y_train,x_test,y_test):

    x_train = x_train.reshape(-1, Length, 4)
    y_train = to_categorical(y_train, num_classes=3)
    x_test = x_test.reshape(-1, Length, 4)
    y_test = to_categorical(y_test, num_classes=3)

    early_stopping = EarlyStopping(monitor='val_loss', patience=3, verbose=1, restore_best_weights=True)

    print("======================")
    print('Convolution Neural Network')
    start_time = time.time()
    model = cnn_classifier()
    model.fit(x_train, y_train, epochs=40, batch_size=50, validation_data=(x_test, y_test), callbacks=[early_stopping])
    #model.save('CNN.h5')
    loss,accuracy = model.evaluate(x_test,y_test)
    print('testing accuracy: {}'.format(accuracy))
    print('training on CPU took %fs'%(time.time()-start_time))


def main():
    x_train,y_train,x_test,y_test = load_data()

    training_process(x_train,y_train,x_test,y_test)


if __name__ == '__main__':
    main()

(6695, 1600)
Convolution Neural Network


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/40
108/108 ━━━━━━━━━━━━━━━━━━━━ 8s 62ms/step - accuracy: 0.9790 - loss: 0.0822 - val_accuracy: 1.0000 - val_loss: 4.4455e-04
Epoch 2/40
108/108 ━━━━━━━━━━━━━━━━━━━━ 11s 74ms/step - accuracy: 0.9988 - loss: 0.0089 - val_accuracy: 1.0000 - val_loss: 0.0017
Epoch 3/40
108/108 ━━━━━━━━━━━━━━━━━━━━ 8s 50ms/step - accuracy: 0.9970 - loss: 0.0175 - val_accuracy: 1.0000 - val_loss: 0.0023
Epoch 4/40
108/108 ━━━━━━━━━━━━━━━━━━━━ 7s 62ms/step - accuracy: 0.9973 - loss: 0.0095 - val_accuracy: 1.0000 - val_loss: 6.7345e-04
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.
42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 4.4610e-04
testing accuracy: 1.0
training on CPU took 34.801978s
